# IHMM Demonstration Notebook

This notebook demonstrates how to use the IHMM implementation for fixation detection, including:
- Generating synthetic data
- Running the IHMM pipeline
- Visualizing results


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import polars as pl

import pymovements as pm
# import your functions
from pymovements.events.detection.ihmm import compute_hmm, ihmm
from pymovements.gaze.experiment import Experiment

%config InlineBackend.figure_format = 'svg'

## Load Toy Dataset

In [ ]:
# Define the experimental setup
experiment = Experiment(
    screen_width_px=1280,
    screen_height_px=1024,
    screen_width_cm=38,
    screen_height_cm=30.2,
    distance_cm=68,
    origin="upper left",
    sampling_rate=250.0,
)

# Load gaze data from a CSV file and initialize three Gaze objects to test different modes.
gaze1 = pm.gaze.from_csv(
    "./gaze-toy-example.csv",
    experiment=experiment,
    time_column="time",
    pixel_columns=["x", "y"],
)

gaze2 = pm.gaze.from_csv(
    "./gaze-toy-example.csv",
    experiment=experiment,
    time_column="time",
    pixel_columns=["x", "y"],
)
gaze3 = pm.gaze.from_csv(
    "./gaze-toy-example.csv",
    experiment=experiment,
    time_column="time",
    pixel_columns=["x", "y"],
)
gaze4 = pm.gaze.from_csv(
    "./gaze-toy-example.csv",
    experiment=experiment,
    time_column="time",
    pixel_columns=["x", "y"],
)
gaze5 = pm.gaze.from_csv(
    "./gaze-toy-example.csv",
    experiment=experiment,
    time_column="time",
    pixel_columns=["x", "y"],
)

# Convert pixel coordinates to degrees of visual angle (dva).
# Requires a valid Experiment with screen geometry and distance.
gaze1.pix2deg()
gaze1.pos2vel()

gaze2.pix2deg()
gaze2.pos2vel()

gaze3.pix2deg()
gaze3.pos2vel()

gaze4.pix2deg()
gaze4.pos2vel()

gaze5.pix2deg()
gaze5.pos2vel()

gaze1.detect("idt", name="idt_fixations")

gaze1

# The function parameters are as such
```
def ihmm(
        velocities: list[list[float]] | list[tuple[float, float]] | np.ndarray,
        timesteps: list[int] | np.ndarray | None = None,
        minimum_duration: int = 100,
        mu: list[float] | np.ndarray | None = None,
        sigma: list[float] | np.ndarray | None = None,
        init_state: list[float] | np.ndarray | None = None,
        transition_probabilities: list[list[float]] | np.ndarray | None = None,
        reestimation_max_iters: int = 1000,
        reestimation: bool = False,
        verbose: bool = False,
        hmm_parameters_dict: dict | None = None,
        name: str = 'fixation',
) -> Events:
```

## Run IHMM with default parameters

In [ ]:
gaze1.detect("ihmm", name="ihmm_fixations")

gaze1.events

# Run IHMM with reestimation

In [ ]:
# Use the "initialization" parameter to run the Baum-Welch
# algorithm to obtain optimal starting parameters

# Flag "verbose" as True to print and see the optimal obtained parameters

# The parameter "reestimation_max_iters" is used to stop the Baum-Welch
# algorithm if it runs too long,

# it should be need only for very large datasets as the algorithm usually
# reaches convergence quickly

gaze2.detect(
    "ihmm",
    reestimation=True,
    verbose=True,
    reestimation_max_iters=1000,
    name="ihmm_fixations")

gaze2.events

# Run IHMM with custom HMM parameters

In [ ]:
# Input the different values manually

mu = [10, 300]

sigma = [11, 386]

init = [0.5, 0.5]

trans = [[0.95, 0.05], [0.05, 0.95]]


gaze3.detect(
    "ihmm",
    mu=mu,
    sigma=sigma,
    init_state=init,
    transition_probabilities=trans,
    name="fixation_ihmm")


# Or use a dictionary to reuse the reestimated parameters

dict = {'mu': [2.0140785987072283,
               69.41529375180232],
        'sigma': [1.3220152347857583,
                  87.32409626093241],
        'init': [0.9999999999992323,
                 1.000000000000001e-12],
        'trans': [[0.9736050693754137,
                   0.02639493062458634],
                  [0.07593546563400998,
                   0.9240645343659899]]}

gaze4.detect("ihmm", hmm_parameters_dict=dict, name="ihmm_fixations")

gaze4.events

# Use a custom minimum duration threshold (in milliseconds)

In [ ]:
# Ihmm is very sensitive to any change of states so very low values should
# be able to detect microssacades

# here very low one to see an increase in detected events
gaze5.detect("ihmm", hmm_parameters_dict=dict, minimum_duration=2, name="ihmm_fixations")

gaze5.events

## Visualize Detected events

In [ ]:
# visualize the data
pm.plotting.traceplot(gaze1)

# Plot with deafult values

In [ ]:
gaze1.compute_event_properties(("location", {"position_column": "pixel"}))

pm.plotting.scanpathplot(gaze1, event_name="ihmm_fixations")

# Plot with reestimation

In [ ]:
gaze2.compute_event_properties(("location", {"position_column": "pixel"}))

pm.plotting.scanpathplot(gaze2, event_name="ihmm_fixations")

# Plot with very low minimum duration

In [ ]:
gaze5.compute_event_properties(("location", {"position_column": "pixel"}))
pm.plotting.scanpathplot(gaze5, event_name="ihmm_fixations")

# Plot idt detected fixations

In [ ]:
pm.plotting.scanpathplot(gaze1, event_name="idt_fixations")